# Fidelity/Error Alignment Reanalysis

Post-hoc CPU analysis of the completed Qwen-9B, Gemma-12B, and Qwen-27B stress-prediction experiment.

[实验设置与运行说明](../../../docs/experiment_notes/12_cross_study_audits_and_pilots/c28a7dd105cc6e7e.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 13 tracked cells.')


## 1. Load the frozen three-model stress experiment


In [ ]:
%%cell_progress 1/13 Locate the completed VisualTorque results
cell_stage('Locating the project on Drive or local disk')
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, ModuleNotFoundError):
    print('Not running in Colab; using local project candidates.')

PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
    Path.cwd(), Path.cwd() / 'VisualTorque', Path.cwd().parent,
)
PROJECT_ROOT = next(
    (
        candidate.resolve() for candidate in PROJECT_CANDIDATES
        if (
            (candidate / 'results' / 'model_adaptive_fidelity_stress_prediction_v1').is_dir()
            or (candidate / 'toolkit' / 'pyproject.toml').is_file()
        )
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find the VisualTorque project. In Colab, place it at '
    '/content/drive/MyDrive/VisualTorque.'
)
print('Project root:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/13 Verify or repair the CPU analysis environment
import json
import os
import signal
import subprocess
import sys

probe_code = '''
import json
import matplotlib
import numpy
import pandas
import tqdm
print(json.dumps({
    'numpy': numpy.__version__,
    'pandas': pandas.__version__,
    'matplotlib': matplotlib.__version__,
    'tqdm': tqdm.__version__,
}, sort_keys=True))
'''

probe = subprocess.run(
    [sys.executable, '-c', probe_code],
    text=True, capture_output=True, check=False,
)
if probe.returncode != 0:
    cell_stage('Repairing an inconsistent NumPy/Pandas installation')
    packages = (
        'numpy==2.5.1', 'pandas==3.0.5',
        'matplotlib==3.11.1', 'tqdm==4.67.1',
    )
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        '--no-cache-dir', '--force-reinstall', '--only-binary=:all:',
        *packages,
    ])
    repaired = subprocess.run(
        [sys.executable, '-c', probe_code],
        text=True, capture_output=True, check=False,
    )
    if repaired.returncode != 0:
        raise RuntimeError(repaired.stderr[-4000:])
    print(
        'The analysis stack was repaired. Colab will restart once. '
        'After reconnection, choose Runtime > Run all.',
        flush=True,
    )
    os.kill(os.getpid(), signal.SIGKILL)

cell_stage('Importing the verified analysis stack')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
print(probe.stdout.strip().splitlines()[-1])


In [ ]:
%%cell_progress 3/13 Freeze the exploratory reanalysis protocol
cell_stage('Declaring targets, controls, and output paths')
import hashlib
import json

SEED = 20261124
BOOTSTRAP_REPLICATES = 20000
SIGN_FLIP_REPLICATES = 20000
ALPHA = 0.05

MODEL_SPECS = {
    'qwen9b': {
        'slug': 'Qwen__Qwen3.5-9B',
        'label': 'Qwen-9B',
        'primary_site': 'options',
    },
    'gemma12b': {
        'slug': 'google__gemma-4-12B-it',
        'label': 'Gemma-12B',
        'primary_site': 'answer',
    },
    'qwen27b': {
        'slug': 'Qwen__Qwen3.5-27B__373612cb',
        'label': 'Qwen-27B',
        'primary_site': 'answer',
    },
}
PRIMARY_CELLS = (
    'weight_conflict', 'distance_conflict',
    'max_vs_sum', 'lexicographic_conflict',
)
TARGET_STRATEGY = {
    'weight_conflict': 'weight_only',
    'distance_conflict': 'distance_only',
    'max_vs_sum': 'max_contribution',
    'lexicographic_conflict': 'lexicographic',
}
WRONG_CELL = {
    'weight_conflict': 'distance_conflict',
    'distance_conflict': 'weight_conflict',
    'max_vs_sum': 'lexicographic_conflict',
    'lexicographic_conflict': 'max_vs_sum',
}
PROMPT_STYLES = ('semantic', 'nonce')
MARGIN_BANDS = ('low', 'high')

SOURCE_ROOT = (
    PROJECT_ROOT / 'results'
    / 'model_adaptive_fidelity_stress_prediction_v1'
)
MANIFEST_PATH = SOURCE_ROOT / 'stress_family_manifest.jsonl'
FROZEN_PROTOCOL_PATH = SOURCE_ROOT / 'frozen_protocol.json'
OUTPUT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'fidelity_error_alignment_reanalysis_v1'
)
FIGURE_DIR = OUTPUT_ROOT / 'figures'
for path in (OUTPUT_ROOT, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)

protocol = {
    'schema_version': 'fidelity-error-alignment-reanalysis-1.0',
    'analysis_status': 'post_hoc_exploratory_after_failed_primary',
    'frozen_primary_decision': 'generic_correctness_prediction_not_confirmed',
    'models': list(MODEL_SPECS),
    'primary_cells': list(PRIMARY_CELLS),
    'new_target': (
        'whether the base-scene prediction equals the predeclared shortcut '
        'answer for that diagnostic cell'
    ),
    'specificity_control': 'route fidelity from the paired wrong diagnostic cell',
    'held_out_unit': 'model x diagnostic cell x prompt style x margin band',
    'calibration_reliability': (
        'replicate-0 versus replicate-1 split half and leave-one-family-out'
    ),
    'claim_boundary': (
        'Exploratory reanalysis only. Any positive pattern requires a new, '
        'independent confirmatory manifest.'
    ),
    'source_manifest_sha256': sha256_file(MANIFEST_PATH),
    'source_protocol_sha256': sha256_file(FROZEN_PROTOCOL_PATH),
}
write_json_atomic(OUTPUT_ROOT / 'reanalysis_protocol.json', protocol)
display(protocol)


In [ ]:
%%cell_progress 4/13 Load and validate all three completed model runs
cell_stage('Reading the shared manifest and model-level CSV files')

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

manifest = read_jsonl(MANIFEST_PATH)
assert len(manifest) == 192
assert {row['diagnostic_cell'] for row in manifest}.issuperset(PRIMARY_CELLS)

required_files = (
    'extrapolation_prediction_rows.csv',
    'clean_role_behavior.csv',
    'calibration_route_effects.csv',
    'calibration_congruent_transport.csv',
    'calibration_equal_net_states.csv',
    'outcome_test_families.csv',
    'capture_config.json',
    'model_stress_summary.json',
)
model_roots = {
    key: SOURCE_ROOT / 'models' / specification['slug']
    for key, specification in MODEL_SPECS.items()
}
tables = {}
prediction_parts = []
for model_key, root in model_roots.items():
    missing = [name for name in required_files if not (root / name).is_file()]
    assert not missing, f'{model_key} is missing: {missing}'
    capture = json.loads((root / 'capture_config.json').read_text(encoding='utf-8'))
    identity = capture['identity']
    assert identity['manifest_sha256'] == protocol['source_manifest_sha256']
    assert identity['protocol_sha256'] == protocol['source_protocol_sha256']
    tables[model_key] = {
        name.removesuffix('.csv'): pd.read_csv(root / name)
        for name in required_files if name.endswith('.csv')
    }
    part = tables[model_key]['extrapolation_prediction_rows'].copy()
    assert part['model_key'].eq(model_key).all()
    prediction_parts.append(part)

all_predictions = pd.concat(prediction_parts, ignore_index=True)
all_predictions.insert(0, 'row_id', np.arange(len(all_predictions)))
assert len(all_predictions) == 192
assert all_predictions['prediction_state_id'].nunique() == 48
assert all_predictions.groupby('prediction_state_id').size().eq(4).all()
assert set(all_predictions['diagnostic_cell']) == set(PRIMARY_CELLS)
print('Rows / held-out states:', len(all_predictions), '/', all_predictions['prediction_state_id'].nunique())
print('Models:', sorted(all_predictions['model_key'].unique()))


## 2. Separate exact answers, predeclared shortcuts, and other errors

A generic error label mixes several failure causes.


In [ ]:
%%cell_progress 5/13 Reconstruct exact and shortcut response labels
cell_stage('Mapping every test prediction to exact, target-shortcut, or other')

manifest_rows = []
replicate_rows = []
for row in manifest:
    replicate_rows.append({
        'family_id': row['family_id'],
        'stratum_replicate': int(row['stratum_replicate']),
        'split': row['split'],
    })
    if row['split'] != 'outcome_test' or row['diagnostic_cell'] not in PRIMARY_CELLS:
        continue
    features = row['roles']['base']['features']
    target_strategy = TARGET_STRATEGY[row['diagnostic_cell']]
    manifest_rows.append({
        'family_id': row['family_id'],
        'diagnostic_cell': row['diagnostic_cell'],
        'target_strategy': target_strategy,
        'exact_semantic': features['strategy_predictions']['exact_torque'],
        'shortcut_semantic': features['strategy_predictions'][target_strategy],
    })
manifest_target = pd.DataFrame(manifest_rows)
assert len(manifest_target) == 64
assert (manifest_target['exact_semantic'] != manifest_target['shortcut_semantic']).all()
replicate_frame = pd.DataFrame(replicate_rows)

behavior_parts = []
for model_key in MODEL_SPECS:
    frame = tables[model_key]['clean_role_behavior']
    frame = frame[
        (frame['split'] == 'outcome_test')
        & (frame['role'] == 'base')
        & frame['diagnostic_cell'].isin(PRIMARY_CELLS)
    ][['model_key', 'family_id', 'prediction']].copy()
    behavior_parts.append(frame)
base_predictions = pd.concat(behavior_parts, ignore_index=True)
assert len(base_predictions) == 192

strategy_rows = all_predictions.merge(
    manifest_target,
    on=['family_id', 'diagnostic_cell'],
    how='left', validate='many_to_one',
).merge(
    base_predictions,
    on=['model_key', 'family_id'],
    how='left', validate='one_to_one',
)
strategy_rows['exact_hit'] = (
    strategy_rows['prediction'] == strategy_rows['exact_semantic']
).astype(int)
strategy_rows['shortcut_adoption'] = (
    strategy_rows['prediction'] == strategy_rows['shortcut_semantic']
).astype(int)
strategy_rows['response_class'] = np.select(
    [
        strategy_rows['exact_hit'].eq(1),
        strategy_rows['shortcut_adoption'].eq(1),
    ],
    ['exact', 'target_shortcut'],
    default='other',
)
assert (strategy_rows['exact_hit'] == strategy_rows['base_correct']).all()
assert not (
    strategy_rows['exact_hit'].eq(1)
    & strategy_rows['shortcut_adoption'].eq(1)
).any()
strategy_rows.to_csv(OUTPUT_ROOT / 'strategy_aligned_outcome_rows.csv', index=False)
display(pd.crosstab(
    strategy_rows['model_key'], strategy_rows['response_class'], normalize='index'
).round(3))


In [ ]:
%%cell_progress 6/13 Define fixed state-held-out prediction utilities
cell_stage('Defining dependency-light logistic regression and state-level inference')

FEATURE_SETS = (
    'metadata', 'behavior',
    'wrong_calibration_mechanism', 'matched_calibration_mechanism',
)
CATEGORICAL_LEVELS = {
    'model_key': tuple(MODEL_SPECS),
    'diagnostic_cell': PRIMARY_CELLS,
    'prompt_style': PROMPT_STYLES,
    'margin_band': MARGIN_BANDS,
    'orientation': ('left', 'right'),
    'structure': ('single', 'multi'),
}
BASE_NUMERIC = ('normalized_margin', 'log_total_torque', 'stack_count')
NUMERIC_FEATURES = {
    'metadata': BASE_NUMERIC,
    'behavior': BASE_NUMERIC + (
        'clean_top1_gap', 'clean_normalized_entropy', 'prior_inrange_accuracy',
    ),
    'wrong_calibration_mechanism': BASE_NUMERIC + (
        'clean_top1_gap', 'clean_normalized_entropy', 'prior_inrange_accuracy',
        'wrong_calibration_route_fidelity',
    ),
    'matched_calibration_mechanism': BASE_NUMERIC + (
        'clean_top1_gap', 'clean_normalized_entropy', 'prior_inrange_accuracy',
        'calibration_route_fidelity',
    ),
}

def categorical_matrix(frame):
    columns = []
    for column, levels in CATEGORICAL_LEVELS.items():
        values = frame[column].astype(str).to_numpy()
        assert set(values).issubset(set(levels)), (column, set(values))
        for level in levels[1:]:
            columns.append((values == level).astype(float))
    return np.column_stack(columns)

def design_matrices(train, test, feature_set):
    train_cat = categorical_matrix(train)
    test_cat = categorical_matrix(test)
    numeric = NUMERIC_FEATURES[feature_set]
    train_num = train[list(numeric)].to_numpy(dtype=float)
    test_num = test[list(numeric)].to_numpy(dtype=float)
    means = train_num.mean(axis=0)
    scales = train_num.std(axis=0)
    scales[scales < 1e-8] = 1.0
    train_num = (train_num - means) / scales
    test_num = (test_num - means) / scales
    return (
        np.column_stack([np.ones(len(train)), train_cat, train_num]),
        np.column_stack([np.ones(len(test)), test_cat, test_num]),
    )

def sigmoid(values):
    return 1.0 / (1.0 + np.exp(-np.clip(values, -30, 30)))

def fit_logistic(x, y, l2=1.0, max_iter=100):
    y = np.asarray(y, dtype=float)
    beta = np.zeros(x.shape[1], dtype=float)
    penalty = np.eye(x.shape[1], dtype=float) * l2
    penalty[0, 0] = 0.0
    for _ in range(max_iter):
        probability = sigmoid(x @ beta)
        weight = np.clip(probability * (1 - probability), 1e-6, None)
        gradient = x.T @ (y - probability) - penalty @ beta
        information = x.T @ (x * weight[:, None]) + penalty
        step = np.linalg.solve(information, gradient)
        beta += step
        if np.max(np.abs(step)) < 1e-8:
            break
    return beta

def auc_score(y, score):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    positives = int(y.sum())
    negatives = len(y) - positives
    if positives == 0 or negatives == 0:
        return np.nan
    ranks = pd.Series(score).rank(method='average').to_numpy()
    return float(
        (ranks[y == 1].sum() - positives * (positives + 1) / 2)
        / (positives * negatives)
    )

def run_leave_one_state_out(frame, target_column, label):
    rows = []
    states = sorted(frame['prediction_state_id'].unique())
    for state_id in tqdm(states, desc=label, unit='state'):
        test_mask = frame['prediction_state_id'] == state_id
        train = frame.loc[~test_mask]
        test = frame.loc[test_mask]
        y_train = train[target_column].to_numpy(dtype=float)
        for feature_set in FEATURE_SETS:
            train_x, test_x = design_matrices(train, test, feature_set)
            if len(np.unique(y_train)) == 1:
                probability = np.full(len(test), np.clip(y_train.mean(), 1e-6, 1 - 1e-6))
            else:
                beta = fit_logistic(train_x, y_train)
                probability = np.clip(sigmoid(test_x @ beta), 1e-6, 1 - 1e-6)
            for row_id, target, value in zip(
                test['row_id'], test[target_column], probability
            ):
                rows.append({
                    'row_id': int(row_id),
                    'prediction_state_id': state_id,
                    'feature_set': feature_set,
                    'target': int(target),
                    'probability': float(value),
                })
    result = pd.DataFrame(rows)
    result['log_loss'] = -(
        result['target'] * np.log(result['probability'])
        + (1 - result['target']) * np.log(1 - result['probability'])
    )
    result['brier'] = (result['probability'] - result['target']) ** 2
    return result

def prediction_metrics(oof):
    rows = []
    for feature_set, group in oof.groupby('feature_set'):
        rows.append({
            'feature_set': feature_set,
            'n': int(len(group)),
            'log_loss': float(group['log_loss'].mean()),
            'brier': float(group['brier'].mean()),
            'auroc': auc_score(group['target'], group['probability']),
        })
    return pd.DataFrame(rows)

def paired_state_test(values, seed):
    values = np.asarray(values, dtype=float)
    generator = np.random.default_rng(seed)
    indices = generator.integers(
        0, len(values), size=(BOOTSTRAP_REPLICATES, len(values))
    )
    distribution = values[indices].mean(axis=1)
    signs = generator.choice(
        (-1.0, 1.0), size=(SIGN_FLIP_REPLICATES, len(values))
    )
    null = (signs * values[None, :]).mean(axis=1)
    estimate = float(values.mean())
    return {
        'states': int(len(values)),
        'estimate': estimate,
        'ci_low': float(np.quantile(distribution, .025)),
        'ci_high': float(np.quantile(distribution, .975)),
        'sign_flip_p': float(
            (1 + np.sum(np.abs(null) >= abs(estimate)))
            / (SIGN_FLIP_REPLICATES + 1)
        ),
    }

def summarize_prediction(oof, frame, prefix):
    metrics = prediction_metrics(oof)
    state_metric = oof.groupby([
        'prediction_state_id', 'feature_set'
    ])[['log_loss', 'brier']].mean().reset_index()
    state_log = state_metric.pivot(
        index='prediction_state_id', columns='feature_set', values='log_loss'
    )
    state_brier = state_metric.pivot(
        index='prediction_state_id', columns='feature_set', values='brier'
    )
    specifications = (
        ('logloss_behavior_minus_matched',
         state_log['behavior'] - state_log['matched_calibration_mechanism']),
        ('logloss_wrong_minus_matched',
         state_log['wrong_calibration_mechanism']
         - state_log['matched_calibration_mechanism']),
        ('brier_behavior_minus_matched',
         state_brier['behavior'] - state_brier['matched_calibration_mechanism']),
    )
    tests = pd.DataFrame([
        {'comparison': name, **paired_state_test(values.to_numpy(), SEED + index)}
        for index, (name, values) in enumerate(specifications)
    ])

    # Correct mapping: preserve every unique state before selecting metadata.
    state_meta = (
        frame[['prediction_state_id', 'diagnostic_cell', 'model_key']]
        .drop_duplicates('prediction_state_id')
        .set_index('prediction_state_id')
    )
    assert len(state_meta) == 48
    improvement = pd.DataFrame({
        'prediction_state_id': state_log.index,
        'behavior_minus_matched_logloss': (
            state_log['behavior'] - state_log['matched_calibration_mechanism']
        ).to_numpy(),
    }).merge(
        state_meta.reset_index(), on='prediction_state_id', validate='one_to_one'
    )
    cell_robustness = improvement.groupby('diagnostic_cell').agg(
        states=('prediction_state_id', 'size'),
        mean_logloss_improvement=('behavior_minus_matched_logloss', 'mean'),
    ).reset_index()
    model_robustness = improvement.groupby('model_key').agg(
        states=('prediction_state_id', 'size'),
        mean_logloss_improvement=('behavior_minus_matched_logloss', 'mean'),
    ).reset_index()
    assert cell_robustness['states'].eq(12).all()
    assert model_robustness['states'].eq(16).all()

    oof.to_csv(OUTPUT_ROOT / f'{prefix}_oof_predictions.csv', index=False)
    metrics.to_csv(OUTPUT_ROOT / f'{prefix}_prediction_metrics.csv', index=False)
    tests.to_csv(OUTPUT_ROOT / f'{prefix}_state_tests.csv', index=False)
    cell_robustness.to_csv(
        OUTPUT_ROOT / f'{prefix}_cell_robustness.csv', index=False
    )
    model_robustness.to_csv(
        OUTPUT_ROOT / f'{prefix}_model_robustness.csv', index=False
    )
    return metrics, tests, cell_robustness, model_robustness

print('Prediction utilities ready; no scikit-learn dependency is used.')


In [ ]:
%%cell_progress 7/13 Correct and freeze the original correctness analysis
cell_stage('Recomputing the failed primary analysis with all 48 states')
correctness_oof = run_leave_one_state_out(
    all_predictions, 'base_correct', 'Generic correctness prediction'
)
(
    correctness_metrics, correctness_tests,
    correctness_cell_robustness, correctness_model_robustness,
) = summarize_prediction(
    correctness_oof, all_predictions, 'generic_correctness'
)

behavior_test = correctness_tests.set_index('comparison').loc[
    'logloss_behavior_minus_matched'
]
correctness_confirmed = bool(
    behavior_test['ci_low'] > 0 and behavior_test['sign_flip_p'] < ALPHA
)
assert not correctness_confirmed, (
    'Unexpected reversal of the frozen primary decision; audit inputs and code.'
)
display(correctness_metrics.round(4))
display(correctness_tests.round(4))
display(correctness_cell_robustness.round(4))
display(correctness_model_robustness.round(4))
print('Frozen decision: generic correctness prediction was NOT confirmed.')


In [ ]:
%%cell_progress 8/13 Test alignment with the predeclared shortcut answer
cell_stage('Predicting specific shortcut adoption rather than generic error')
shortcut_oof = run_leave_one_state_out(
    strategy_rows, 'shortcut_adoption', 'Shortcut-adoption prediction'
)
(
    shortcut_metrics, shortcut_tests,
    shortcut_cell_robustness, shortcut_model_robustness,
) = summarize_prediction(
    shortcut_oof, strategy_rows, 'shortcut_adoption'
)

response_summary = strategy_rows.groupby([
    'model_key', 'diagnostic_cell', 'prompt_style', 'margin_band'
]).agg(
    n=('family_id', 'size'),
    accuracy=('exact_hit', 'mean'),
    shortcut_rate=('shortcut_adoption', 'mean'),
    other_rate=('response_class', lambda values: np.mean(values == 'other')),
    route_fidelity=('calibration_route_fidelity', 'first'),
).reset_index()
assert response_summary['n'].eq(4).all()
response_summary.to_csv(
    OUTPUT_ROOT / 'strategy_state_summary.csv', index=False
)

shortcut_lookup = shortcut_tests.set_index('comparison')
shortcut_signal = bool(
    shortcut_lookup.loc['logloss_behavior_minus_matched', 'ci_low'] > 0
    and shortcut_lookup.loc['logloss_wrong_minus_matched', 'ci_low'] > 0
)
display(shortcut_metrics.round(4))
display(shortcut_tests.round(4))
display(shortcut_cell_robustness.round(4))
display(shortcut_model_robustness.round(4))
print('Exploratory matched-and-specific signal:', shortcut_signal)


## 3. Ask whether the causal score itself is stable

Every route state was estimated from four calibration families.


In [ ]:
%%cell_progress 9/13 Measure calibration reliability
cell_stage('Comparing two balanced calibration halves and leave-one-family-out estimates')

route_parts = []
for model_key, specification in MODEL_SPECS.items():
    frame = tables[model_key]['calibration_route_effects'].copy()
    frame = frame[
        (frame['site_role'] == specification['primary_site'])
        & frame['diagnostic_cell'].isin(PRIMARY_CELLS)
    ].merge(
        replicate_frame[['family_id', 'stratum_replicate']],
        on='family_id', how='left', validate='many_to_one',
    )
    assert frame['stratum_replicate'].isin([0, 1]).all()
    control = tables[model_key]['calibration_congruent_transport']
    control = control[
        control['site_role'] == specification['primary_site']
    ]['exact_transport_effect'].mean()
    assert abs(control) > 1e-6
    frame['model_key'] = model_key
    frame['control_efficacy'] = float(control)
    route_parts.append(frame)
calibration_rows = pd.concat(route_parts, ignore_index=True)
state_columns = [
    'model_key', 'diagnostic_cell', 'prompt_style', 'margin_band'
]
assert calibration_rows.groupby(state_columns).size().eq(4).all()

split_half = calibration_rows.groupby(
    state_columns + ['stratum_replicate']
).agg(
    families=('family_id', 'size'),
    raw_effect=('direct_causal_effect', 'mean'),
    control_efficacy=('control_efficacy', 'first'),
).reset_index()
split_half['fidelity'] = (
    split_half['raw_effect'] / split_half['control_efficacy']
)
split_wide = split_half.pivot(
    index=state_columns, columns='stratum_replicate', values='fidelity'
).reset_index().rename(columns={0: 'half_0', 1: 'half_1'})
assert len(split_wide) == 48

def safe_correlation(left, right):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    if len(left) < 3 or np.std(left) < 1e-12 or np.std(right) < 1e-12:
        return np.nan
    return float(np.corrcoef(left, right)[0, 1])

def spearman(left, right):
    return safe_correlation(
        pd.Series(left).rank(method='average'),
        pd.Series(right).rank(method='average'),
    )

reliability_rows = [{
    'group': 'all_48_states',
    'states': len(split_wide),
    'pearson': safe_correlation(split_wide['half_0'], split_wide['half_1']),
    'spearman': spearman(split_wide['half_0'], split_wide['half_1']),
    'sign_agreement': float(
        (np.sign(split_wide['half_0']) == np.sign(split_wide['half_1'])).mean()
    ),
    'mean_absolute_difference': float(
        np.abs(split_wide['half_0'] - split_wide['half_1']).mean()
    ),
}]
for model_key, group in split_wide.groupby('model_key'):
    reliability_rows.append({
        'group': f'model::{model_key}',
        'states': len(group),
        'pearson': safe_correlation(group['half_0'], group['half_1']),
        'spearman': spearman(group['half_0'], group['half_1']),
        'sign_agreement': float(
            (np.sign(group['half_0']) == np.sign(group['half_1'])).mean()
        ),
        'mean_absolute_difference': float(
            np.abs(group['half_0'] - group['half_1']).mean()
        ),
    })
split_reliability = pd.DataFrame(reliability_rows)

loo_rows = []
for keys, group in calibration_rows.groupby(state_columns, sort=True):
    values = group['direct_causal_effect'].to_numpy(dtype=float)
    denominator = float(group['control_efficacy'].iloc[0])
    estimates = np.asarray([
        np.delete(values, index).mean() / denominator
        for index in range(len(values))
    ])
    full = float(values.mean() / denominator)
    signs = np.sign(estimates)
    loo_rows.append({
        **dict(zip(state_columns, keys)),
        'families': len(values),
        'full_fidelity': full,
        'loo_mean': float(estimates.mean()),
        'loo_sd': float(estimates.std(ddof=1)),
        'loo_range': float(estimates.max() - estimates.min()),
        'loo_sign_agreement_with_full': float((signs == np.sign(full)).mean()),
    })
loo_frame = pd.DataFrame(loo_rows)
assert len(loo_frame) == 48
split_wide.to_csv(OUTPUT_ROOT / 'calibration_split_half_states.csv', index=False)
split_reliability.to_csv(
    OUTPUT_ROOT / 'calibration_split_half_reliability.csv', index=False
)
loo_frame.to_csv(OUTPUT_ROOT / 'calibration_leave_one_out_states.csv', index=False)
display(split_reliability.round(4))
display(loo_frame.groupby('model_key')[[
    'loo_sd', 'loo_range', 'loo_sign_agreement_with_full'
]].mean().round(4))


In [ ]:
%%cell_progress 10/13 Compare equal-net invariance and route transport
cell_stage('Testing two limits of the scalar fidelity interpretation')

equal_behavior_parts = []
equal_causal_parts = []
for model_key, specification in MODEL_SPECS.items():
    behavior = tables[model_key]['outcome_test_families']
    behavior = behavior[
        behavior['diagnostic_cell'] == 'equal_torque_recomposition'
    ].groupby(['prompt_style', 'margin_band']).agg(
        test_families=('family_id', 'size'),
        behavioral_same_net_stability=('same_net_stable', 'mean'),
        behavioral_accuracy=('base_correct', 'mean'),
    ).reset_index()
    behavior['model_key'] = model_key
    equal_behavior_parts.append(behavior)

    causal = tables[model_key]['calibration_equal_net_states']
    causal = causal[
        causal['site_role'] == specification['primary_site']
    ][[
        'prompt_style', 'margin_band', 'causal_invariance_fidelity',
        'invariant_tvd', 'mirror_tvd',
    ]].copy()
    causal['model_key'] = model_key
    equal_causal_parts.append(causal)

equal_alignment = pd.concat(equal_behavior_parts).merge(
    pd.concat(equal_causal_parts),
    on=['model_key', 'prompt_style', 'margin_band'],
    validate='one_to_one',
)
assert len(equal_alignment) == 12
equal_correlations = {
    'states': len(equal_alignment),
    'pearson': safe_correlation(
        equal_alignment['causal_invariance_fidelity'],
        equal_alignment['behavioral_same_net_stability'],
    ),
    'spearman': spearman(
        equal_alignment['causal_invariance_fidelity'],
        equal_alignment['behavioral_same_net_stability'],
    ),
}
equal_alignment.to_csv(OUTPUT_ROOT / 'equal_net_alignment_states.csv', index=False)
write_json_atomic(OUTPUT_ROOT / 'equal_net_alignment_summary.json', equal_correlations)

transport = all_predictions.groupby([
    'model_key', 'diagnostic_cell'
]).agg(
    new_extrapolation_route=('calibration_route_fidelity', 'mean'),
    prior_inrange_route=('prior_route_fidelity', 'first'),
).reset_index()
transport['same_sign'] = (
    np.sign(transport['new_extrapolation_route'])
    == np.sign(transport['prior_inrange_route'])
)
transport.to_csv(OUTPUT_ROOT / 'prior_to_extrapolation_route_transport.csv', index=False)
transport_summary = {
    'model_cell_pairs': len(transport),
    'same_sign_pairs': int(transport['same_sign'].sum()),
    'same_sign_rate': float(transport['same_sign'].mean()),
    'pearson': safe_correlation(
        transport['prior_inrange_route'], transport['new_extrapolation_route']
    ),
    'spearman': spearman(
        transport['prior_inrange_route'], transport['new_extrapolation_route']
    ),
}
write_json_atomic(OUTPUT_ROOT / 'route_transport_summary.json', transport_summary)
display(equal_alignment.round(3))
display(equal_correlations)
display(transport.round(3))
display(transport_summary)


In [ ]:
%%cell_progress 11/13 Save the paper-facing diagnostic figure
cell_stage('Rendering corrected prediction, strategy, and reliability panels')

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
feature_order = [
    'metadata', 'behavior',
    'wrong_calibration_mechanism', 'matched_calibration_mechanism',
]
feature_labels = ['Metadata', 'Behavior', 'Wrong route', 'Matched route']
colors = ['#64748B', '#7C3AED', '#D97706', '#2563EB']

correct_plot = correctness_metrics.set_index('feature_set').loc[feature_order]
axes[0, 0].bar(np.arange(4), correct_plot['log_loss'], color=colors)
axes[0, 0].set_xticks(np.arange(4), feature_labels, rotation=20, ha='right')
axes[0, 0].set_ylabel('Held-out log loss (lower is better)')
axes[0, 0].set_title('A. Generic correctness: frozen negative result')

shortcut_plot = shortcut_metrics.set_index('feature_set').loc[feature_order]
axes[0, 1].bar(np.arange(4), shortcut_plot['log_loss'], color=colors)
axes[0, 1].set_xticks(np.arange(4), feature_labels, rotation=20, ha='right')
axes[0, 1].set_ylabel('Held-out log loss (lower is better)')
axes[0, 1].set_title('B. Specific shortcut adoption (exploratory)')

response_plot = pd.crosstab(
    strategy_rows['model_key'], strategy_rows['response_class'], normalize='index'
).reindex(index=list(MODEL_SPECS), columns=['exact', 'target_shortcut', 'other'], fill_value=0)
bottom = np.zeros(len(response_plot))
for response_class, color in zip(
    ['exact', 'target_shortcut', 'other'], ['#0F766E', '#DC2626', '#94A3B8']
):
    values = response_plot[response_class].to_numpy()
    axes[1, 0].bar(
        np.arange(len(values)), values, bottom=bottom,
        label=response_class.replace('_', ' '), color=color,
    )
    bottom += values
axes[1, 0].set_xticks(
    np.arange(len(MODEL_SPECS)),
    [MODEL_SPECS[key]['label'] for key in MODEL_SPECS],
)
axes[1, 0].set_ylim(0, 1)
axes[1, 0].set_ylabel('Fraction of stress cases')
axes[1, 0].set_title('C. What kind of answer did the model choose?')
axes[1, 0].legend(frameon=False)

for model_key, group in split_wide.groupby('model_key'):
    axes[1, 1].scatter(
        group['half_0'], group['half_1'], s=55, alpha=.8,
        label=MODEL_SPECS[model_key]['label'],
    )
limits = np.asarray([
    split_wide[['half_0', 'half_1']].min().min(),
    split_wide[['half_0', 'half_1']].max().max(),
])
axes[1, 1].plot(limits, limits, '--', color='black', linewidth=1)
axes[1, 1].set_xlabel('Calibration half 0')
axes[1, 1].set_ylabel('Calibration half 1')
axes[1, 1].set_title('D. Four-family route-score reliability')
axes[1, 1].legend(frameon=False)

fig.suptitle('VisualTorque: what the failed prediction test does and does not mean', fontweight='bold')
fig.tight_layout()
figure_path = FIGURE_DIR / 'fidelity_error_alignment_reanalysis.png'
fig.savefig(figure_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)


In [ ]:
%%cell_progress 12/13 Freeze the reanalysis decision and report
cell_stage('Writing an auditable claim-boundary report')

correct_lookup = correctness_tests.set_index('comparison')
shortcut_lookup = shortcut_tests.set_index('comparison')
pooled_reliability = split_reliability.iloc[0].to_dict()

decision = {
    'schema_version': 'fidelity-error-alignment-reanalysis-decision-1.0',
    'analysis_status': 'post_hoc_exploratory',
    'generic_correctness_status': 'not_confirmed',
    'generic_correctness_behavior_minus_matched': correct_lookup.loc[
        'logloss_behavior_minus_matched'
    ].to_dict(),
    'shortcut_alignment_signal': shortcut_signal,
    'shortcut_behavior_minus_matched': shortcut_lookup.loc[
        'logloss_behavior_minus_matched'
    ].to_dict(),
    'shortcut_wrong_minus_matched': shortcut_lookup.loc[
        'logloss_wrong_minus_matched'
    ].to_dict(),
    'calibration_split_half': pooled_reliability,
    'equal_net_alignment': equal_correlations,
    'route_transport': transport_summary,
    'corrected_state_counts': {
        'total': 48,
        'per_diagnostic_cell': 12,
        'per_model': 16,
    },
    'interpretation': (
        'The original prospective claim remains failed. The shortcut target, '
        'calibration reliability, and equal-net alignment are exploratory '
        'diagnostics that determine whether a new confirmatory experiment is '
        'worth designing.'
    ),
}
write_json_atomic(OUTPUT_ROOT / 'reanalysis_decision.json', decision)

def interval_text(row):
    return (
        f"{row['estimate']:.4f} [{row['ci_low']:.4f}, "
        f"{row['ci_high']:.4f}], p={row['sign_flip_p']:.4g}"
    )

report = [
    '# Fidelity/Error Alignment Reanalysis', '',
    '## Frozen conclusion', '',
    'The prospective generic-correctness claim remains **not confirmed**. '
    'This notebook does not redefine that target after seeing the data.', '',
    '- Behavior minus matched-route log loss: ' + interval_text(
        correct_lookup.loc['logloss_behavior_minus_matched']
    ) + '.', '',
    '## Corrected robustness accounting', '',
    '- Total held-out states: 48.',
    '- States per diagnostic cell: 12 (3 models × 2 prompts × 2 margins).',
    '- States per model: 16 (4 cells × 2 prompts × 2 margins).',
    '- The prior table reported 3 and 4 because metadata was deduplicated '
    'after the state identifier had been moved into the index.', '',
    '## Exploratory shortcut-alignment target', '',
    f'- Matched-and-specific signal: **{shortcut_signal}**.',
    '- Behavior minus matched-route log loss: ' + interval_text(
        shortcut_lookup.loc['logloss_behavior_minus_matched']
    ) + '.',
    '- Wrong-route minus matched-route log loss: ' + interval_text(
        shortcut_lookup.loc['logloss_wrong_minus_matched']
    ) + '.', '',
    'This target asks whether the model selected the particular shortcut '
    'answer declared before scoring each conflict family. It does not collapse '
    'visual-reading failures, answer-format errors, alternative shortcuts, and '
    'physics-composition failures into one generic error label.', '',
    '## Calibration reliability', '',
    f"- Split-half Pearson: {pooled_reliability['pearson']:.3f}.",
    f"- Split-half Spearman: {pooled_reliability['spearman']:.3f}.",
    f"- Split-half sign agreement: {pooled_reliability['sign_agreement']:.3f}.",
    '- Each half contains one left-facing and one right-facing calibration '
    'family for every state.', '',
    '## Equal-net and support-shift diagnostics', '',
    f"- Causal/behavioral equal-net Spearman across 12 states: "
    f"{equal_correlations['spearman']:.3f}.",
    f"- Prior-to-extrapolation route signs retained: "
    f"{transport_summary['same_sign_pairs']}/{transport_summary['model_cell_pairs']}.",
    '', '## Claim boundary', '',
    'All analyses after the frozen generic-correctness test are exploratory. '
    'If shortcut alignment is both predictive and task-specific, the next step '
    'is a new scene-disjoint confirmation set with the shortcut-adoption target '
    'registered in advance. Otherwise the paper should retain the simpler '
    'boundary claim: local causal routes explain interventions on average but '
    'the present scalar summary does not forecast individual stress errors.',
]
(OUTPUT_ROOT / 'REANALYSIS_REPORT.md').write_text(
    '\n'.join(report) + '\n', encoding='utf-8'
)
display(decision)


In [ ]:
%%cell_progress 13/13 Validate and inventory the reanalysis artifacts
cell_stage('Checking row counts, decision boundaries, and saved files')

required_outputs = (
    'reanalysis_protocol.json',
    'strategy_aligned_outcome_rows.csv',
    'generic_correctness_prediction_metrics.csv',
    'generic_correctness_state_tests.csv',
    'generic_correctness_cell_robustness.csv',
    'shortcut_adoption_prediction_metrics.csv',
    'shortcut_adoption_state_tests.csv',
    'calibration_split_half_states.csv',
    'calibration_split_half_reliability.csv',
    'calibration_leave_one_out_states.csv',
    'equal_net_alignment_states.csv',
    'prior_to_extrapolation_route_transport.csv',
    'reanalysis_decision.json',
    'REANALYSIS_REPORT.md',
)
missing = [name for name in required_outputs if not (OUTPUT_ROOT / name).is_file()]
assert not missing, missing
assert correctness_cell_robustness['states'].eq(12).all()
assert correctness_model_robustness['states'].eq(16).all()
assert decision['generic_correctness_status'] == 'not_confirmed'

print('Reanalysis complete. No model or GPU was used.')
print('Decision:', OUTPUT_ROOT / 'reanalysis_decision.json')
print('Report:', OUTPUT_ROOT / 'REANALYSIS_REPORT.md')
print('Figure:', FIGURE_DIR / 'fidelity_error_alignment_reanalysis.png')
